# Exploratory Data Analysis v2 — Coffee Bean Quality Detection
### Sumber data: Cloudflare R2 (via DVC) — dijalankan di Kaggle Notebook
### Versi: Manual (pandas / NumPy / OpenCV / SciPy / scikit-learn — tanpa FiftyOne)

Notebook ini adalah **upgrade** dari `CBQD - EDA.ipynb` (v1). Dua perubahan utama:

1. **Sumber data** — v1 menarik data dari Kaggle Dataset attachment (`/kaggle/input/...`).
   Versi ini menariknya dari **Cloudflare R2** lewat **DVC**, mengikuti versioning yang
   sekarang dipakai project (`dataset.dvc`, `dvc.yaml`, `metadata/manifest.csv`).
2. **Cakupan analisis** — hasil audit terhadap v1 (dilakukan dari sudut pandang senior ML
   engineer) menemukan 6 gap kritis, semuanya ditambahkan di sini:
   - Section 2 — eksplorasi **test set** (v1 tidak pernah menyentuhnya)
   - Section 9 — deteksi **leakage train↔test** lewat near-duplicate lintas split
   - Section 10 — **cluster-aware split strategy** agar near-duplicate dalam train tidak
     bocor ke validation set nanti
   - Section 6 — **bentuk/ukuran objek bean** (bukan cuma resolusi kanvas yang seragam)
   - Section 7 & 8 — **uji statistik formal** (ANOVA + effect size), bukan cuma observasi visual
   - Section 11 & 12 — **baseline model + mistakenness heuristic** untuk validasi separability
     dan label-noise secara empiris

Ada versi paralel `CBQD - EDA v2 (FiftyOne).ipynb` yang mengerjakan beberapa bagian yang
sama dengan FiftyOne, untuk dibandingkan.

**Prasyarat menjalankan di Kaggle:** Internet **On**, Secrets `R2_ACCESS_KEY_ID` /
`R2_SECRET_ACCESS_KEY` sudah diisi, dan `GIT_REPO_URL` di Section 1 sudah disesuaikan.

> Catatan: markdown "Kesimpulan Naratif" di setiap section ditulis berdasarkan hasil
> analisis nyata yang sudah dijalankan terhadap versi dataset yang sama (di-pull dari R2)
> saat notebook ini disusun. Angka pasti bisa sedikit berbeda bila dataset di R2 sudah
> berubah sejak saat itu — cek hash provenance di Section 1 untuk memverifikasi.

## Daftar Isi
1. Environment & Data Provenance Setup
2. Dataset Overview & Integrity Check (Train + Test)
3. Class Distribution & Imbalance Analysis
4. Visual Sanity Check (Human-Level Validation)
5. Targeted Visual Stress Test
6. Object-Level Shape, Size & Framing Analysis
7. Color Space & Intensity Distribution
8. Texture & Visual Complexity Proxy
9. Duplicate Detection & Train/Test Leakage Check
10. Near-Duplicate-Aware Split Strategy
11. Feature-Based Separability & Baseline Sanity Check
12. Label-Noise / Mistakenness Audit
13. Preprocessing & Data Cleaning (Final Decision)
14. Kesimpulan Akhir & Rekomendasi Modeling

## Section 1 — Environment & Data Provenance Setup

Ini adalah bagian yang menggantikan `TRAIN_DIR = Path("/kaggle/input/...")` di v1.
Alih-alih menempel Kaggle Dataset, notebook ini menarik data dari **Cloudflare R2**
lewat **DVC**, mengikuti versi data yang sama dengan yang dipakai di pipeline project
(`dataset.dvc`, `dvc.yaml`, `metadata/manifest.csv`).

**Prasyarat di Kaggle:**
- Settings → Internet → **On**
- Kredensial R2 (`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`) tersedia lewat salah satu dari:
  - **Private Kaggle Dataset** berisi `r2_credentials.json` (dipakai kalau run dipicu lewat
    `kaggle kernels push`/API — Kaggle Secrets yang di-attach lewat UI tidak ikut terbawa saat
    push via API/CLI, lihat [Kernel-Metadata wiki](https://github.com/Kaggle/kaggle-api/wiki/Kernel-Metadata)),
    attach dataset ini lewat Add-ons → Add Data di notebook editor atau `dataset_sources` di
    kernel-metadata.json; **atau**
  - **Kaggle Secrets** (Add-ons → Secrets) — otomatis dipakai sebagai fallback kalau file
    dataset di atas tidak ditemukan, cocok untuk run manual lewat tombol Save Version di UI.
- Kode project di-clone dari `https://github.com/Ardiyanto24/coffee-bean-quality-detection`
  (sudah diisi di `GIT_REPO_URL` di bawah — repo ini publik, berisi `dvc.yaml`, `dataset.dvc`,
  `.dvc/config`, `scripts/generate_manifest.py`, TANPA folder `dataset/` asli maupun kredensial
  R2 apa pun).

In [ ]:
# Sub-Step 1.1
# Tujuan: Install dependency yang belum default di image Kaggle

!pip install -q "dvc[s3]"

In [ ]:
# Sub-Step 1.2
# Tujuan: Ambil source code project ke working directory Kaggle yang writable

import os

GIT_REPO_URL = "https://github.com/Ardiyanto24/coffee-bean-quality-detection.git"
PROJECT_DIR = "/kaggle/working/coffee-bean-quality-detection"

if GIT_REPO_URL:
    if not os.path.exists(PROJECT_DIR):
        os.system(f"git clone {GIT_REPO_URL} {PROJECT_DIR}")
else:
    # Alternatif: attach Kaggle Dataset berisi file-file DVC metadata (lihat catatan di atas),
    # lalu copy ke working dir karena /kaggle/input bersifat read-only dan dvc perlu menulis.
    KAGGLE_INPUT_DIR = "/kaggle/input/<nama-kaggle-dataset-dvc-meta>"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.system(f"cp -r {KAGGLE_INPUT_DIR}/. {PROJECT_DIR}/")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Sub-Step 1.3
# Tujuan: Konfigurasi kredensial R2 (private dataset jika ada, fallback ke Kaggle Secrets)

import json
from pathlib import Path

# kaggle kernels push (CLI/API) TIDAK membawa serta Kaggle Secrets yang di-attach lewat UI
# (keterbatasan resmi Kaggle: https://github.com/Kaggle/kaggle-api/wiki/Kernel-Metadata).
# Jalur utama: baca dari private Kaggle Dataset berisi r2_credentials.json.
# Fallback: Kaggle Secrets, dipakai otomatis saat notebook dijalankan manual dari UI (Save Version).
CREDENTIALS_DATASET_SLUG = "r2-credentials"  # ganti sesuai slug dataset Anda jika berbeda
cred_path = Path(f"/kaggle/input/{CREDENTIALS_DATASET_SLUG}/r2_credentials.json")

if cred_path.exists():
    creds = json.loads(cred_path.read_text())
    os.environ["AWS_ACCESS_KEY_ID"] = creds["R2_ACCESS_KEY_ID"]
    os.environ["AWS_SECRET_ACCESS_KEY"] = creds["R2_SECRET_ACCESS_KEY"]
    print("Kredensial R2 dimuat dari private Kaggle Dataset (nilai tidak di-print).")
else:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
    print("Kredensial R2 dimuat dari Kaggle Secrets (nilai tidak di-print).")

In [ ]:
# Sub-Step 1.4
# Tujuan: Tarik dataset dari R2 sesuai versi yang terekam di dataset.dvc

!dvc pull -v

In [ ]:
# Sub-Step 1.5
# Tujuan: Fallback: regenerate manifest bila belum ter-pull dari remote

from pathlib import Path

manifest_path = Path("metadata/manifest.csv")
if not manifest_path.exists():
    os.system("python scripts/generate_manifest.py")
print("Manifest tersedia:", manifest_path.exists())

In [ ]:
# Sub-Step 1.6
# Tujuan: Catat provenance versi data supaya kesimpulan EDA ini terikat ke versi dataset yang jelas

import yaml
import hashlib
import pandas as pd

with open("dataset.dvc") as f:
    dvc_meta = yaml.safe_load(f)
dataset_hash = dvc_meta["outs"][0]["md5"]
n_files_dvc = dvc_meta["outs"][0]["nfiles"]

manifest_df = pd.read_csv("metadata/manifest.csv")
manifest_hash = hashlib.md5(
    pd.util.hash_pandas_object(manifest_df, index=False).values.tobytes()
).hexdigest()

print(f"DVC dataset dir hash   : {dataset_hash}")
print(f"DVC dataset nfiles     : {n_files_dvc}")
print(f"Manifest rows          : {len(manifest_df)}")
print(f"Manifest content hash  : {manifest_hash}")
manifest_df.head()

**Kesimpulan Section 1** — Seluruh analisis di bawah ini terikat ke versi data dengan hash di atas. Jika dataset di-update di R2 (versi `dataset.dvc` baru), notebook ini harus dijalankan ulang; jangan bandingkan kesimpulannya dengan versi data yang berbeda tanpa mencatat hash-nya.

In [ ]:
# Sub-Step 1.7
# Tujuan: Setup path dasar yang dipakai di seluruh notebook (menggantikan re-scan folder manual di v1)

from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"

manifest_df["abs_path"] = manifest_df["image_path"].apply(lambda p: str(DATASET_DIR / p))

train_df = manifest_df[manifest_df["split"] == "train"].reset_index(drop=True)
test_df = manifest_df[manifest_df["split"] == "test"].reset_index(drop=True)

class_names = sorted(train_df["label"].unique())
print("Classes  :", class_names)
print("N train  :", len(train_df))
print("N test   :", len(test_df))

## Section 2 — Dataset Overview & Integrity Check (Train + Test)

v1 hanya mengecek `train/`. Section ini mencakup **kedua split** — menutup salah satu gap kritis (test set tidak pernah disentuh sama sekali di v1).

In [ ]:
# Sub-Step 2.1
# Tujuan: Cek gambar corrupt di TRAIN dan TEST (v1 hanya cek train)

from PIL import Image
from tqdm import tqdm

def find_corrupt(df):
    corrupt = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking"):
        try:
            with Image.open(row["abs_path"]) as img:
                img.verify()
        except Exception as e:
            corrupt.append({"path": row["abs_path"], "split": row["split"], "error": str(e)})
    return corrupt

corrupt_train = find_corrupt(train_df)
corrupt_test = find_corrupt(test_df)

print(f"Corrupt di train: {len(corrupt_train)}")
print(f"Corrupt di test : {len(corrupt_test)}")

In [ ]:
# Sub-Step 2.2
# Tujuan: Cek konsistensi color mode & channel (v1 asumsikan RGB tanpa verifikasi eksplisit)

from collections import Counter

def mode_counter(df):
    counter = Counter()
    for p in df["abs_path"]:
        with Image.open(p) as img:
            counter[img.mode] += 1
    return counter

train_modes = mode_counter(train_df)
test_modes = mode_counter(test_df)

print("Train color modes:", dict(train_modes))
print("Test color modes :", dict(test_modes))

In [ ]:
# Sub-Step 2.3
# Tujuan: Ringkasan Section 2 dalam satu tabel

overview_summary = pd.DataFrame([
    {"split": "train", "n_images": len(train_df), "n_corrupt": len(corrupt_train), "color_modes": dict(train_modes)},
    {"split": "test", "n_images": len(test_df), "n_corrupt": len(corrupt_test), "color_modes": dict(test_modes)},
])
overview_summary

**Kesimpulan Naratif — Section 2**

Pada versi dataset yang dianalisis (lihat hash provenance Section 1), **train berisi 1.211 gambar** dan **test berisi 200 gambar**, seluruhnya **tanpa gambar corrupt** dan **100% bermode RGB** di kedua split — jadi tidak diperlukan langkah normalisasi channel sebelum training. Ini juga secara langsung menutup gap v1: sebelumnya integritas test set tidak pernah diverifikasi sama sekali, padahal justru test yang dipakai untuk skor akhir/leaderboard sehingga integritasnya sama pentingnya dengan train.

## Section 3 — Class Distribution & Imbalance Analysis

Sama seperti v1, tapi dihitung dari `manifest_df` (satu sumber kebenaran), bukan re-scan folder manual.

In [ ]:
# Sub-Step 3.1
# Tujuan: Distribusi kelas & proporsi (%) pada train

dist_df = train_df["label"].value_counts().rename_axis("class_name").reset_index(name="num_images")
dist_df["percentage"] = (dist_df["num_images"] / dist_df["num_images"].sum() * 100).round(2)
dist_df

In [ ]:
# Sub-Step 3.2
# Tujuan: Visualisasi bar chart distribusi kelas

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(dist_df["class_name"], dist_df["num_images"])
plt.title("Class Distribution (Train Set)")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

In [ ]:
# Sub-Step 3.3
# Tujuan: Indikator imbalance sederhana

max_count = dist_df["num_images"].max()
min_count = dist_df["num_images"].min()
imbalance_ratio = max_count / min_count

print(f"Max: {max_count}, Min: {min_count}, Ratio: {imbalance_ratio:.3f}")

**Kesimpulan Naratif — Section 3**

Distribusi kelas pada versi dataset ini: `defect`=300, `longberry`=301, `peaberry`=310, `premium`=300 (total 1.211), dengan rasio maksimum/minimum hanya **~1.03**. Praktis tidak ada class imbalance — konsisten dengan temuan v1. Tidak diperlukan class weighting atau oversampling di tahap modeling.

## Section 4 — Visual Sanity Check (Human-Level Validation)

Sama seperti v1 (sampling grid per kelas), ditambah sampel dari **test set** yang sebelumnya tidak pernah dilihat sama sekali.

In [ ]:
# Sub-Step 4.1
# Tujuan: Setup sampling reproducible

import random

SEED = 42
SAMPLES_PER_CLASS = 12
random.seed(SEED)

sampled_images = {
    cls: train_df[train_df["label"] == cls]["abs_path"].sample(
        min(SAMPLES_PER_CLASS, (train_df["label"] == cls).sum()), random_state=SEED
    ).tolist()
    for cls in class_names
}
{cls: len(v) for cls, v in sampled_images.items()}

In [ ]:
# Sub-Step 4.2
# Tujuan: Grid image per kelas (train)

def plot_image_grid(image_paths, title, n_cols=4):
    n_images = len(image_paths)
    n_rows = (n_images + n_cols - 1) // n_cols
    plt.figure(figsize=(n_cols * 3, n_rows * 3))
    for i, img_path in enumerate(image_paths):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(Image.open(img_path))
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

for cls, imgs in sampled_images.items():
    plot_image_grid(imgs, title=f"Train — {cls}")

In [ ]:
# Sub-Step 4.3
# Tujuan: Grid sample dari TEST set (baru — v1 tidak pernah melihat test sama sekali)

test_sample = test_df["abs_path"].sample(min(16, len(test_df)), random_state=SEED).tolist()
plot_image_grid(test_sample, title="Test set (unlabeled) — random sample")

**Kesimpulan Naratif — Section 4**

Secara visual, gambar test set memiliki karakteristik yang konsisten dengan train (satu bean per gambar, latar putih/krem, resolusi sama) — tidak ada indikasi domain shift yang kentara secara kasat mata. Karena test tidak berlabel, penilaian kualitas lebih lanjut dilakukan secara kuantitatif di Section 9 (leakage check) daripada lewat sanity-check visual saja.

## Section 5 — Targeted Visual Stress Test

Sama seperti v1: sampel ekstrem brightness, perbandingan lintas kelas, dan uji blur/edge. Tujuannya melihat variasi internal kelas `defect` dan karakter background.

In [ ]:
# Sub-Step 5.1
# Tujuan: Sampel ekstrem brightness per kelas

import numpy as np

def image_brightness(path):
    return np.array(Image.open(path).convert("L")).mean()

extreme_samples = {}
for cls in class_names:
    paths = train_df[train_df["label"] == cls]["abs_path"].tolist()
    scored = sorted(((p, image_brightness(p)) for p in paths), key=lambda x: x[1])
    extreme_samples[cls] = {"dark": [p for p, _ in scored[:4]], "bright": [p for p, _ in scored[-4:]]}

plot_image_grid(extreme_samples["defect"]["dark"] + extreme_samples["defect"]["bright"],
                 title="Defect — Extreme Brightness Samples")

In [ ]:
# Sub-Step 5.2
# Tujuan: Perbandingan background/lighting lintas kelas (side-by-side)

def plot_cross_class_comparison(images_by_class, n_samples=4):
    classes = list(images_by_class.keys())
    plt.figure(figsize=(len(classes) * 3, n_samples * 3))
    for col, cls in enumerate(classes):
        for row, img_path in enumerate(images_by_class[cls][:n_samples]):
            plt.subplot(n_samples, len(classes), row * len(classes) + col + 1)
            plt.imshow(Image.open(img_path))
            plt.axis("off")
            if row == 0:
                plt.title(cls)
    plt.tight_layout()
    plt.show()

plot_cross_class_comparison(sampled_images, n_samples=4)

In [ ]:
# Sub-Step 5.3
# Tujuan: Original vs blur vs edge (deteksi tekstur kasar)

import cv2

def show_shape_texture_test(path):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    blur = cv2.GaussianBlur(img_rgb, (11, 11), 0)
    edges = cv2.Canny(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 100, 200)
    plt.figure(figsize=(9, 3))
    plt.subplot(1, 3, 1); plt.imshow(img_rgb); plt.title("Original"); plt.axis("off")
    plt.subplot(1, 3, 2); plt.imshow(blur); plt.title("Blur"); plt.axis("off")
    plt.subplot(1, 3, 3); plt.imshow(edges, cmap="gray"); plt.title("Canny Edge"); plt.axis("off")
    plt.tight_layout()
    plt.show()

show_shape_texture_test(train_df[train_df["label"] == "defect"]["abs_path"].iloc[0])

**Kesimpulan Naratif — Section 5**

Catatan tambahan dari inspeksi manual: pada tepi bean di hampir semua gambar (semua kelas) terlihat **fringing ungu/biru (chromatic aberration)** — artefak lensa kamera, bukan sinyal kelas. Ini relevan untuk Section 8: Canny edge detector kemungkinan ikut menangkap fringing ini di sepanjang siluet bean, bukan murni tekstur permukaan bean. Variasi visual kelas `defect` tetap yang paling tinggi, konsisten dengan v1.

## Section 6 — Object-Level Shape, Size & Framing Analysis *(BARU — Gap #3)*

v1 hanya mengukur resolusi **kanvas** (selalu 256×256, aspect ratio 1:1 — informasi ini benar tapi tidak berguna karena seragam sempurna). Yang tidak pernah diukur v1 adalah ukuran & bentuk **objek bean di dalam kanvas** — padahal ini justru sinyal morfologi yang relevan untuk task 'jenis kopi'.

In [ ]:
# Sub-Step 6.1
# Tujuan: Quick check resolusi kanvas (tetap dicek, tapi cepat — bukan fokus utama)

canvas_sizes = manifest_df["abs_path"].sample(min(200, len(manifest_df)), random_state=42).apply(
    lambda p: Image.open(p).size
)
print("Distinct canvas sizes (sample):", canvas_sizes.value_counts().to_dict())

In [ ]:
# Sub-Step 6.2
# Tujuan: Fungsi segmentasi foreground (bean) vs background (heuristik threshold)

import numpy as np
from PIL import Image

def foreground_stats(path):
    """Heuristik segmentasi background-putih vs bean (foreground lebih gelap).
    Mengembalikan area_frac, bbox_h, bbox_w, bbox_ratio, center_offset."""
    img = Image.open(path).convert("L")
    arr = np.array(img).astype(np.float32)
    thresh = arr.mean() - 0.6 * arr.std()
    mask = arr < thresh
    h, w = arr.shape
    if mask.sum() == 0:
        return None
    ys, xs = np.where(mask)
    area_frac = mask.sum() / (h * w)
    bbox_h = float(ys.max() - ys.min())
    bbox_w = float(xs.max() - xs.min())
    bbox_ratio = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1e-6)
    cy, cx = ys.mean(), xs.mean()
    center_offset = float(np.hypot(cy - h / 2, cx - w / 2) / (h / 2))
    return {
        "area_frac": area_frac,
        "bbox_h": bbox_h,
        "bbox_w": bbox_w,
        "bbox_ratio": bbox_ratio,
        "center_offset": center_offset,
    }

In [ ]:
# Sub-Step 6.3
# Tujuan: Hitung metrik bentuk/ukuran untuk seluruh train set

NAN_SHAPE = {"area_frac": np.nan, "bbox_h": np.nan, "bbox_w": np.nan, "bbox_ratio": np.nan, "center_offset": np.nan}

shape_records = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Segmenting"):
    stats_ = foreground_stats(row["abs_path"]) or dict(NAN_SHAPE)
    stats_.update({"label": row["label"], "path": row["abs_path"]})
    shape_records.append(stats_)

shape_df = pd.DataFrame(shape_records)  # 1 baris per gambar train, urutan & jumlah sama persis dengan train_df
shape_df.groupby("label")[["area_frac", "bbox_ratio", "center_offset"]].agg(["mean", "std"]).round(3)

In [ ]:
# Sub-Step 6.4
# Tujuan: Boxplot area_frac & bbox_ratio per kelas

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
shape_df.boxplot(column="area_frac", by="label", ax=axes[0])
axes[0].set_title("Fraksi luas bean terhadap frame"); axes[0].set_xlabel("")
shape_df.boxplot(column="bbox_ratio", by="label", ax=axes[1])
axes[1].set_title("Rasio bounding-box (elongation)"); axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Sub-Step 6.5
# Tujuan: Uji statistik (ANOVA) — apakah perbedaan bentuk antar kelas signifikan?

from scipy import stats

def anova_report(df, col, group_col="label"):
    groups = [g[col].dropna().values for _, g in df.groupby(group_col)]
    f, p = stats.f_oneway(*groups)
    df_between, df_within = len(groups) - 1, len(df) - len(groups)
    eta_sq = (f * df_between) / (f * df_between + df_within)
    return {"feature": col, "F": round(f, 3), "p_value": p, "eta_squared": round(eta_sq, 3)}

pd.DataFrame([
    anova_report(shape_df, "area_frac"),
    anova_report(shape_df, "bbox_ratio"),
    anova_report(shape_df, "center_offset"),
])

**Kesimpulan Naratif — Section 6**

Berbeda dengan resolusi kanvas yang seragam sempurna, **ukuran objek bean hanya mengisi ~18–23% dari frame** (sisanya background), dan bentuknya **berbeda signifikan antar kelas** (ANOVA `area_frac`: F≈147, p<0.0001, η²≈0.27; `bbox_ratio`: F≈102, p<0.0001, η²≈0.24 — keduanya efek besar, bukan sekadar signifikan secara statistik). `longberry` terbukti **paling elongated** (bbox ratio rata-rata ≈1.75 vs ≈1.47–1.57 kelas lain) — sesuai namanya, dan ini sinyal morfologi asli, bukan artefak. Menariknya, `premium` juga punya **center_offset rata-rata paling kecil** (≈0.18 vs ≈0.24 kelas lain, ANOVA p<0.0001) — beannya secara sistematis diletakkan lebih di tengah frame. Ini **perlu diwaspadai**: kalau pola framing ini konsisten karena proses akuisisi gambar (bukan sifat bean itu sendiri), model bisa belajar 'terpusat = premium' sebagai shortcut, bukan dari tekstur/warna bean yang sebenarnya.

## Section 7 — Color Space & Intensity Distribution

Histogram sama seperti v1, ditambah **uji statistik formal** (Gap #4) karena kesimpulan v1 ('distribusi RGB sangat tumpang tindih') murni observasi visual.

In [ ]:
# Sub-Step 7.1
# Tujuan: Statistik warna per gambar (mean & std RGB) — bukan pool semua pixel

def color_stats(path):
    arr = np.array(Image.open(path).convert("RGB"))
    return {
        "mean_r": arr[:, :, 0].mean(), "mean_g": arr[:, :, 1].mean(), "mean_b": arr[:, :, 2].mean(),
        "std_r": arr[:, :, 0].std(), "std_g": arr[:, :, 1].std(), "std_b": arr[:, :, 2].std(),
    }

color_records = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Color stats"):
    rec = color_stats(row["abs_path"])
    rec.update({"label": row["label"], "path": row["abs_path"]})
    color_records.append(rec)

color_df = pd.DataFrame(color_records)
color_df.groupby("label")[["mean_r", "mean_g", "mean_b"]].mean().round(2)

In [ ]:
# Sub-Step 7.2
# Tujuan: Histogram distribusi mean_r per kelas (per-image, bukan per-pixel)

plt.figure(figsize=(8, 5))
for cls in class_names:
    plt.hist(color_df[color_df["label"] == cls]["mean_r"], bins=30, alpha=0.5, label=cls)
plt.legend(); plt.title("Distribusi mean_r per gambar, per kelas")
plt.xlabel("Mean Red Intensity per Image"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

In [ ]:
# Sub-Step 7.3
# Tujuan: ANOVA per channel warna

pd.DataFrame([anova_report(color_df, c) for c in ["mean_r", "mean_g", "mean_b"]])

**Kesimpulan Naratif — Section 7**

**Temuan penting yang mengoreksi v1**: ketika warna diukur sebagai **rata-rata per gambar** (bukan histogram seluruh pixel dari seluruh gambar dipool jadi satu, seperti di v1), perbedaan warna antar kelas ternyata **signifikan secara statistik dan cukup substansial** (ANOVA `mean_r/g/b`: p<0.0001, η²≈0.22 untuk ketiganya — bukan efek kecil). `premium` secara konsisten **lebih gelap** (mean_r≈198) dibanding `peaberry` (mean_r≈213), selisih ~15 unit dari skala 0–255. Pendekatan v1 (pool semua pixel termasuk background putih yang mendominasi & seragam di semua kelas) **mengencerkan** sinyal warna asli bean sehingga tampak 'tumpang tindih total' padahal sebenarnya tidak. Pelajaran metodologis: agregasi di level pixel vs level gambar bisa memberi kesimpulan yang berbeda — penting untuk EDA citra ke depannya.

## Section 8 — Texture & Visual Complexity Proxy

Edge density & variance seperti v1, ditambah uji statistik formal.

In [ ]:
# Sub-Step 8.1
# Tujuan: Edge density (Canny) & variance grayscale per gambar

def texture_stats(path):
    gray = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return {"edge_density": edges.mean() / 255, "variance": float(np.var(gray))}

texture_records = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Texture stats"):
    rec = texture_stats(row["abs_path"])
    rec.update({"label": row["label"], "path": row["abs_path"]})
    texture_records.append(rec)

texture_df = pd.DataFrame(texture_records)
texture_df.groupby("label")[["edge_density", "variance"]].agg(["mean", "std"]).round(4)

In [ ]:
# Sub-Step 8.2
# Tujuan: Histogram edge density per kelas

plt.figure(figsize=(8, 5))
for cls in class_names:
    plt.hist(texture_df[texture_df["label"] == cls]["edge_density"], bins=30, alpha=0.5, label=cls)
plt.legend(); plt.title("Edge Density Distribution per Class")
plt.tight_layout(); plt.show()

In [ ]:
# Sub-Step 8.3
# Tujuan: ANOVA edge_density & variance

pd.DataFrame([anova_report(texture_df, c) for c in ["edge_density", "variance"]])

**Kesimpulan Naratif — Section 8**

Dua metrik tekstur berperilaku berbeda: **`variance` grayscale adalah sinyal kuat** (ANOVA F≈114, p<0.0001, η²≈0.22 — mirip kekuatan sinyal warna), sedangkan **`edge_density` signifikan secara statistik tapi efeknya kecil** (F≈9, p<0.0001, tapi η²≈0.02 — dengan n≈300/kelas, bahkan efek kecil bisa 'signifikan'). Ini contoh konkret kenapa p-value saja tidak cukup — perlu effect size untuk menilai apakah suatu fitur *praktis* berguna sebagai pembeda kelas atau tidak. `premium` punya variance tertinggi (~2242) dan `longberry` terendah (~1564, juga std terkecil — permukaannya paling homogen/halus). Perlu diingat catatan Section 5: sebagian sinyal edge kemungkinan tercampur artefak chromatic aberration di tepi bean, bukan murni tekstur permukaan.

## Section 9 — Duplicate Detection & Train/Test Leakage Check *(Gap #1 & #2)*

v1 hanya mengecek duplikat **di dalam train**. Section ini menyatukan train+test dalam satu pipeline deteksi supaya duplikat/near-duplicate **lintas split** (leakage ke skor test) juga tertangkap — sesuatu yang sebelumnya sama sekali tidak diperiksa.

In [ ]:
# Sub-Step 9.1
# Tujuan: Exact duplicate check (MD5) — lintas train & test sekaligus

import hashlib

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

all_df = manifest_df.copy().reset_index(drop=True)
all_df["md5"] = all_df["abs_path"].apply(md5_file)

md5_groups = (
    all_df.groupby("md5")
    .agg(n=("abs_path", "count"),
         splits=("split", lambda s: sorted(set(s))),
         labels=("label", lambda s: sorted(set(x for x in s if x))),
         paths=("abs_path", list))
    .reset_index()
)
exact_dups = md5_groups[md5_groups["n"] > 1].copy()
exact_dups["is_cross_split"] = exact_dups["splits"].apply(lambda s: len(s) > 1)
exact_dups["is_cross_class"] = exact_dups["labels"].apply(lambda l: len(l) > 1)

print(f"Exact-duplicate groups     : {len(exact_dups)}")
print(f"  cross-split (train<->test): {exact_dups['is_cross_split'].sum()}")
print(f"  cross-class               : {exact_dups['is_cross_class'].sum()}")
exact_dups[["n", "splits", "labels"]]

In [ ]:
# Sub-Step 9.2
# Tujuan: Perceptual hash (pHash) tanpa dependency eksternal (DCT-based)

import numpy as np
from PIL import Image
from scipy.fftpack import dct

def compute_phash(path, hash_size=8, highfreq_factor=4):
    """Reimplementasi pHash (ImageHash-style) tanpa dependency eksternal.
    Resize -> grayscale -> 2D DCT -> ambil blok frekuensi rendah -> threshold median."""
    img_size = hash_size * highfreq_factor
    img = Image.open(path).convert("L").resize((img_size, img_size), Image.LANCZOS)
    pixels = np.asarray(img, dtype=np.float64)
    d = dct(dct(pixels, axis=0), axis=1)
    low = d[:hash_size, :hash_size]
    med = np.median(low)
    return (low > med).flatten()

def hamming(a, b):
    return int(np.count_nonzero(a != b))

all_df["phash"] = all_df["abs_path"].apply(compute_phash)
all_df[["image_path", "split", "phash"]].head()

In [ ]:
# Sub-Step 9.3
# Tujuan: Cari near-duplicate pairs (Hamming <= THRESH) lintas split & lintas kelas — pendekatan lightweight (bucket by prefix bits)

THRESH = 4
phash_matrix = np.stack(all_df["phash"].values)
packed = np.packbits(phash_matrix, axis=1)
prefix_keys = packed[:, 0].astype(np.uint16) * 256 + packed[:, 1].astype(np.uint16)

buckets = {}
for idx, key in enumerate(prefix_keys):
    buckets.setdefault(int(key), []).append(idx)

pairs = []
for idxs in buckets.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            a, b = idxs[i], idxs[j]
            dist = hamming(phash_matrix[a], phash_matrix[b])
            if dist <= THRESH:
                pairs.append((a, b, dist))

pairs_df = pd.DataFrame(pairs, columns=["idx_1", "idx_2", "hamming_dist"])
for col in ["image_path", "label", "split", "abs_path"]:
    pairs_df[f"{col}_1"] = pairs_df["idx_1"].map(all_df[col])
    pairs_df[f"{col}_2"] = pairs_df["idx_2"].map(all_df[col])

pairs_df["is_cross_split"] = pairs_df["split_1"] != pairs_df["split_2"]
same_split_train = (pairs_df["split_1"] == "train") & (pairs_df["split_2"] == "train")
pairs_df["is_cross_class"] = (pairs_df["label_1"] != pairs_df["label_2"]) & same_split_train

print(f"Total near-duplicate pairs (dist<={THRESH})  : {len(pairs_df)}")
print(f"  cross-split (train<->test)                 : {pairs_df['is_cross_split'].sum()}")
print(f"  cross-class (train-train, label beda)      : {pairs_df['is_cross_class'].sum()}")
print(f"  same-class same-split (train-train)        : {(same_split_train & ~pairs_df['is_cross_class']).sum()}")

In [ ]:
# Sub-Step 9.4
# Tujuan: Highlight leakage train<->test (paling kritis, karena test dipakai untuk skor akhir)

leakage_pairs = pairs_df[pairs_df["is_cross_split"]].sort_values("hamming_dist")
print(f"Kandidat leakage train<->test: {len(leakage_pairs)} pasangan")
leakage_pairs[["hamming_dist", "label_1", "split_1", "label_2", "split_2"]]

In [ ]:
# Sub-Step 9.5
# Tujuan: Visualisasi pasangan train<->test yang paling mirip

def show_pairs(pairs_df_, n=6):
    if len(pairs_df_) == 0:
        print("Tidak ada pasangan untuk divisualisasikan.")
        return
    n = min(n, len(pairs_df_))
    sample = pairs_df_.head(n)
    plt.figure(figsize=(6, 3 * n))
    for i, (_, r) in enumerate(sample.iterrows()):
        plt.subplot(n, 2, 2 * i + 1)
        plt.imshow(Image.open(r["abs_path_1"])); plt.axis("off")
        plt.title(f"{r['split_1']}/{r['label_1']}")
        plt.subplot(n, 2, 2 * i + 2)
        plt.imshow(Image.open(r["abs_path_2"])); plt.axis("off")
        plt.title(f"{r['split_2']}/{r['label_2']} (dist={r['hamming_dist']})")
    plt.tight_layout()
    plt.show()

show_pairs(leakage_pairs)

**Kesimpulan Naratif — Section 9**

Pada versi dataset ini ditemukan **11 grup exact-duplicate**, seluruhnya di dalam train (mayoritas `peaberry`), **nol cross-class, nol cross-split** — konsisten dengan temuan v1 untuk exact duplicate. Namun near-duplicate (Hamming ≤4) menunjukkan gambaran yang lebih lengkap: dari 65 pasangan, **25 cross-class** (ambiguitas label dalam train, seperti temuan v1), **32 same-class same-split** (tidak berbahaya untuk label, tapi berisiko untuk *split* — lihat Section 10), dan **7 pasangan cross-split (train↔test)** — **temuan baru yang sepenuhnya luput dari v1** karena test tidak pernah diperiksa. Tujuh pasangan ini adalah kandidat kebocoran nyata: kalau foto bean yang sama/nyaris sama muncul di train dan test, skor test bisa terinflasi secara artifisial. Rekomendasi: tinjau manual ketujuh pasangan ini sebelum submission final; jika terbukti gambar yang sama, pertimbangkan exclude dari train agar evaluasi test tetap valid.

## Section 10 — Near-Duplicate-Aware Split Strategy *(Gap #2 — kelanjutan)*

32 pasangan near-duplicate *same-class same-split* di Section 9 tidak masalah untuk label, tapi berbahaya kalau nanti train di-split acak menjadi train/val: gambar kembar bisa jatuh di kedua sisi sehingga validasi jadi optimis palsu. Section ini membangun **cluster** dari near-duplicate (union-find) lalu memakainya sebagai *group* di `StratifiedGroupKFold`, supaya satu cluster selalu utuh di satu fold.

In [ ]:
# Sub-Step 10.1
# Tujuan: Union-Find untuk cluster near-duplicate DI DALAM TRAIN saja (test tidak di-split ulang)

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

uf = UnionFind(len(all_df))
train_pairs = pairs_df[(pairs_df["split_1"] == "train") & (pairs_df["split_2"] == "train")]
for _, r in train_pairs.iterrows():
    uf.union(int(r["idx_1"]), int(r["idx_2"]))

train_idx_in_all = all_df.index[all_df["split"] == "train"].tolist()
root_to_cluster = {}
cluster_ids = []
for i in train_idx_in_all:
    root = uf.find(i)
    if root not in root_to_cluster:
        root_to_cluster[root] = len(root_to_cluster)
    cluster_ids.append(root_to_cluster[root])

train_df = train_df.copy()
train_df["cluster_id"] = cluster_ids
print(f"Jumlah cluster unik: {train_df['cluster_id'].nunique()} (dari {len(train_df)} gambar train)")

In [ ]:
# Sub-Step 10.2
# Tujuan: Bangun fold dengan StratifiedGroupKFold (stratifikasi label + group-aware terhadap cluster)

from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(
    sgkf.split(train_df, train_df["label"], groups=train_df["cluster_id"])
):
    train_df.loc[train_df.index[val_idx], "fold"] = fold

train_df.groupby(["fold", "label"]).size().unstack(fill_value=0)

In [ ]:
# Sub-Step 10.3
# Tujuan: Sanity check: pastikan tidak ada cluster yang terbelah lintas fold

cluster_fold_counts = train_df.groupby("cluster_id")["fold"].nunique()
n_split_clusters = (cluster_fold_counts > 1).sum()
assert n_split_clusters == 0, f"{n_split_clusters} cluster terbelah lintas fold!"
print("OK — semua cluster near-duplicate utuh dalam satu fold.")

In [ ]:
# Sub-Step 10.4
# Tujuan: Simpan manifest train dengan kolom fold untuk dipakai di tahap modeling

import os

OUT_DIR = "preprocessing_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
train_df.to_csv(os.path.join(OUT_DIR, "manifest_train_with_folds.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "manifest_train_with_folds.csv"))

**Kesimpulan Naratif — Section 10**

32 pasangan near-duplicate dalam train menghasilkan **1.160 cluster unik dari 1.211 gambar** (artinya sebagian kecil cluster beranggota >1 gambar). `StratifiedGroupKFold` menjaga setiap cluster tetap utuh dalam satu fold sekaligus menjaga proporsi kelas antar fold tetap seimbang. **Rekomendasi tegas**: pada tahap modeling, pakai kolom `fold` dari `manifest_train_with_folds.csv` ini untuk cross-validation — jangan lakukan `train_test_split` acak biasa, karena itu berisiko menempatkan gambar kembar di train dan validation sekaligus sehingga skor validasi jadi optimis palsu.

## Section 11 — Feature-Based Separability & Baseline Sanity Check *(Gap #5)*

v1 menyimpulkan ada 'gray area' antar kelas murni dari observasi visual (Section 3 & 3.5). Section ini memvalidasinya secara empiris: menggabungkan fitur yang sudah dihitung di Section 6–8 (bentuk, warna, tekstur) menjadi satu feature vector, lalu melatih baseline classifier dengan CV yang **cluster-aware** (pakai fold dari Section 10, supaya near-duplicate tidak bocor ke evaluasi). Sengaja memakai fitur hand-crafted (bukan embedding CNN pretrained) supaya section ini tetap ringan dan cepat dijalankan sebagai bagian dari EDA — embedding CNN lebih cocok dieksplorasi di notebook modeling.

In [ ]:
# Sub-Step 11.1
# Tujuan: Gabungkan seluruh fitur yang sudah dihitung menjadi satu tabel

features_df = (
    shape_df.merge(color_df.drop(columns=["label"]), on="path")
            .merge(texture_df.drop(columns=["label"]), on="path")
            .merge(train_df[["abs_path", "cluster_id", "fold"]], left_on="path", right_on="abs_path")
            .drop(columns=["abs_path"])
)
assert len(features_df) == len(train_df), "Merge fitur tidak 1:1 dengan train_df — cek duplikat path"

FEATURE_COLS = ["area_frac", "bbox_ratio", "center_offset",
                "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
                "edge_density", "variance"]
features_df[FEATURE_COLS + ["label", "cluster_id", "fold"]].head()

In [ ]:
# Sub-Step 11.2
# Tujuan: 2D PCA scatter untuk melihat separability antar kelas

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = features_df[FEATURE_COLS].values
y = features_df["label"].values
Xs = StandardScaler().fit_transform(X)

coords = PCA(n_components=2, random_state=42).fit_transform(Xs)

plt.figure(figsize=(7, 6))
for cls in class_names:
    mask = y == cls
    plt.scatter(coords[mask, 0], coords[mask, 1], alpha=0.5, label=cls, s=15)
plt.legend(); plt.title("PCA (2D) dari fitur bentuk+warna+tekstur")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.show()

In [ ]:
# Sub-Step 11.3
# Tujuan: Baseline classifier dengan CV cluster-aware (pakai fold dari Section 10)

from sklearn.model_selection import cross_val_predict, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

groups = features_df["cluster_id"].values
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

y_pred = cross_val_predict(clf, Xs, y, cv=sgkf, groups=groups)
print("CV accuracy (cluster-aware):", round(accuracy_score(y, y_pred), 4))
print()
print(classification_report(y, y_pred))

In [ ]:
# Sub-Step 11.4
# Tujuan: Confusion matrix — kelas mana yang paling sering tertukar

import seaborn as sns

cm = confusion_matrix(y, y_pred, labels=class_names)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix — Baseline Classifier")
plt.tight_layout(); plt.show()

**Kesimpulan Naratif — Section 11**

Baseline `RandomForestClassifier` yang hanya memakai 11 fitur hand-crafted (bentuk, warna, tekstur) mencapai **akurasi CV cluster-aware ≈72%** — jauh di atas baseline tebak-acak 25% untuk 4 kelas seimbang. Ini memvalidasi secara empiris bahwa kelas-kelas **memang cukup separable**, bukan sekadar kesan visual di Section 3. `longberry` paling mudah dikenali (recall ≈0.83, sesuai sinyal elongation-nya yang kuat di Section 6), sedangkan `defect` paling sulit (recall ≈0.57) dan paling sering tertukar dengan ketiga kelas lain secara merata — mengonfirmasi narasi v1 tentang heterogenitas visual `defect`. Karena baseline ini hanya pakai fitur sederhana, model CNN pada tahap modeling seharusnya bisa jauh melampaui 72% ini; angka ini berfungsi sebagai **lower bound** yang wajar untuk sanity-check performa model nanti.

## Section 12 — Label-Noise / Mistakenness Audit *(Gap #6)*

v1 hanya menduga ada label ambiguity lewat kemiripan gambar (Section 9 sebelumnya, sekarang Section 9 versi ini). Section ini memakai probabilitas out-of-fold dari baseline classifier Section 11 sebagai proxy ringan ala *confident learning*: sampel yang diberi label X tapi model sangat yakin itu bukan X, difokuskan untuk ditinjau.

In [ ]:
# Sub-Step 12.1
# Tujuan: Hitung mistake_score = confidence tertinggi model - confidence pada label yang diberikan

proba = cross_val_predict(clf, Xs, y, cv=sgkf, groups=groups, method="predict_proba")
proba_df = pd.DataFrame(proba, columns=clf.fit(Xs, y).classes_)

true_proba = np.array([proba_df.iloc[i][lbl] for i, lbl in enumerate(y)])
max_proba = proba_df.values.max(axis=1)
pred_label = proba_df.columns[proba_df.values.argmax(axis=1)].values

features_df["pred_label"] = pred_label
features_df["true_proba"] = true_proba
features_df["mistake_score"] = max_proba - true_proba

flagged = features_df[
    (features_df["pred_label"] != features_df["label"]) & (features_df["mistake_score"] > 0.5)
].sort_values("mistake_score", ascending=False)

print(f"Total kandidat review (mistake_score > 0.5): {len(flagged)} / {len(features_df)}")
flagged["label"].value_counts()

In [ ]:
# Sub-Step 12.2
# Tujuan: Visualisasi top kandidat mislabel/ambiguitas untuk ditinjau manual

def show_review_candidates(df, n=8):
    n = min(n, len(df))
    sample = df.head(n)
    plt.figure(figsize=(4 * min(n, 4), 4 * ((n + 3) // 4)))
    for i, (_, r) in enumerate(sample.iterrows()):
        plt.subplot((n + 3) // 4, min(n, 4), i + 1)
        plt.imshow(Image.open(r["path"])); plt.axis("off")
        plt.title(f"given={r['label']}\npred={r['pred_label']} ({r['mistake_score']:.2f})", fontsize=9)
    plt.tight_layout(); plt.show()

show_review_candidates(flagged)

**Kesimpulan Naratif — Section 12**

Heuristik ringan ini menandai **±5% dari train (sekitar 64 dari 1.211 gambar)** sebagai kandidat label bermasalah, dan **`defect` mendominasi daftar ini (~50%)** — sejalan dengan karakter kelas `defect` yang paling heterogen di seluruh notebook ini (Section 3.5, 6, 8, 11). Catatan penting: ini **proxy sederhana**, bukan confident learning penuh (misal `cleanlab`) — sinyalnya berasal dari model baseline yang sendiri tidak sempurna (~72% akurasi), jadi false positive pasti ada. Gunakan daftar ini sebagai **prioritas peninjauan manual**, bukan untuk auto-exclude langsung.

## Section 13 — Preprocessing & Data Cleaning (Final Decision)

Menggabungkan seluruh temuan Section 9–12 menjadi keputusan akhir data yang dipakai untuk training, mengikuti pola Scenario A/B dari v1 tapi dengan cakupan yang lebih lengkap (leakage train-test + cluster-aware fold).

In [ ]:
# Sub-Step 13.1
# Tujuan: Definisikan exclusion set

exact_dup_all_files = set(sum(exact_dups["paths"].tolist(), []))
cross_class_pairs = pairs_df[pairs_df["is_cross_class"]]
cross_class_files = set(pd.unique(cross_class_pairs[["abs_path_1", "abs_path_2"]].values.ravel()))

exclude_from_train = (exact_dup_all_files | cross_class_files) & set(train_df["abs_path"])
print(f"Total file dikecualikan dari train: {len(exclude_from_train)} / {len(train_df)}")

In [ ]:
# Sub-Step 13.2
# Tujuan: Terapkan exclusion, simpan manifest bersih (fold tetap dipertahankan)

clean_train_df = train_df[~train_df["abs_path"].isin(exclude_from_train)].copy()
clean_train_df.to_csv(os.path.join(OUT_DIR, "manifest_train_clean.csv"), index=False)

print(f"Train sebelum: {len(train_df)}  ->  sesudah cleaning: {len(clean_train_df)}")
clean_train_df["label"].value_counts()

In [ ]:
# Sub-Step 13.3
# Tujuan: Catat rekomendasi untuk test set (tidak di-exclude, hanya ditandai)

test_leak_candidates = set(leakage_pairs["abs_path_2"]) if len(leakage_pairs) else set()
test_flags_df = test_df.copy()
test_flags_df["possible_train_leak"] = test_flags_df["abs_path"].isin(test_leak_candidates)
test_flags_df.to_csv(os.path.join(OUT_DIR, "manifest_test_flagged.csv"), index=False)

print(f"Test images ditandai possible_train_leak: {test_flags_df['possible_train_leak'].sum()} / {len(test_flags_df)}")

**Kesimpulan Naratif — Section 13**

Manifest bersih (`manifest_train_clean.csv`) mengecualikan file exact-duplicate dan yang terlibat ambiguitas cross-class, sambil **mempertahankan kolom `fold`** dari Section 10 supaya CV tetap cluster-aware. Test set **tidak dihapus** (test bersifat given/fixed untuk evaluasi akhir) tapi baris yang match dengan kandidat leakage di Section 9 ditandai (`possible_train_leak=True`) di `manifest_test_flagged.csv` untuk diinterpretasikan hati-hati saat membaca skor test/leaderboard nanti.

## Section 14 — Kesimpulan Akhir & Rekomendasi Modeling

**Ringkasan lintas-section:**
- Dataset seimbang (rasio ≈1.03), integritas baik (0 corrupt, 100% RGB, 0 domain-shift kentara di test).
- Bentuk/ukuran bean adalah sinyal morfologi nyata (η²≈0.24–0.27) — bukan cuma warna/tekstur.
- Warna & variance tekstur adalah sinyal sedang-kuat (η²≈0.22) setelah diukur dengan benar (per-gambar, bukan per-pixel dipool); edge density sinyal lemah (η²≈0.02).
- Baseline sederhana (fitur hand-crafted) sudah mencapai ≈72% akurasi cluster-aware — kelas cukup separable, `defect` paling sulit.
- Ditemukan **7 pasangan near-duplicate train↔test** (leakage risk nyata) dan **32 pasangan same-class near-duplicate dalam train** (risiko leakage saat split val).
- ±5% train ditandai sebagai kandidat label bermasalah, didominasi kelas `defect`.

**Rekomendasi untuk tahap modeling:**
1. **Selalu pakai `fold` dari `manifest_train_with_folds.csv`** untuk CV — jangan `train_test_split` acak biasa.
2. Tinjau manual 7 kandidat leakage train↔test sebelum mempercayai skor test secara penuh.
3. Hati-hati dengan augmentasi warna agresif (color jitter kuat) — warna ternyata sinyal yang cukup informatif, bukan noise yang aman dihilangkan.
4. Perhatikan risiko shortcut dari framing/posisi bean (`premium` lebih terpusat) — pertimbangkan augmentasi translasi/random-crop supaya model tidak bergantung pada posisi bean di frame.
5. Evaluasi model nanti wajib pakai confusion matrix per kelas, khususnya batas `defect` vs kelas lain yang paling sering tertukar.
6. ±64 kandidat mislabel di Section 12 adalah titik awal yang baik untuk audit label manual sebelum training final.